In [ ]:
import time
import numpy as np
import meshio
import warnings

try:
    import cupy
    CUPY_AVAILABLE = True

except ImportError:
    cupy = None
    CUPY_AVAILABLE = False

    warnings.warn(
        "CuPy is not installed. GPU smoothing is disabled, "
        "but all CPU PacMesh functions remain available.",
        RuntimeWarning,
        stacklevel=2,
    )
    

#Importing PakMsh utilities
import PakMsh
from PakMsh.generationUtils import generate_mesh
from PakMsh.sizingUtils import create_sizing_function
from PakMsh.smoothingUtils import physical_smooth,cvt_smooth_cpu,cvt_smooth_gpu,smart_laplacian_smooth_numba
from PakMsh.plotUtils import plot_mesh,plot_mesh_circles,plot_fast_voronoi
from PakMsh.statUtils import mesh_sizing_check,get_mesh_stats
start_time = time.perf_counter()

##### Input Parameters ######
####################################################
# Rectangle dimensions 
depth_z =  -3000     
length_x = 9200     
# SEGY parameters
fname = "marmousi.segy"
hmin_segy = 0.0  # Minimum Element size for segy, will apply if higher than function minimum
wl=2             # Number of elements per wavelenght
freq=45           # Frequency in hz
# Mesh Generator parameters
npoints = 5000000 # Maximum number of points to generate
N = 2528 #3651 4774 5898 # Function resolution or Mesh grid resolution if fineness is custom
fineness = "custom" # Mesh grid resolution relative to the minimum function value
overlap = 1.25    # How much circle overlap is allowed, 1.1 = 10%
grade = 0.85 # function grading for smooth element transition, 1 = no smooth, 0.1 = very high smooth
physical_iterations = 150 # Number of physical smooth iterations
cvtcpu_iterations = 150 # Number of CVT CPU smooth iterations
cvtgpu_iterations = 150 # Number of CVT GPU smooth iterations
cvtgpu_resolution = 2*N # Number of pixels for CVT GPU
# Size of padding
padding_z = 1500      
padding_x = 3000  
subdomain = True # Enable rectangular subdomain boundaries if True
padding_type = None # rectangular, elliptical, none
#padding_type = "rectangular"
#padding_type = "elliptical"
# Ellipse exponent if elliptical padding
ellipse_n = 3.0
###################################################
# Maximum axis calculation for plotting functions
segy_bbox = (depth_z, 0, 0, length_x)
if(padding_type=="rectangular" or padding_type=="elliptical" ): 
    x_range_total=(- padding_x, length_x + padding_x)
    z_range_total=(0.0, depth_z - padding_z)
    (x_min, x_max), (z_min, z_max) = x_range_total, z_range_total
else:
    x_range_total=(0.0, length_x)
    z_range_total=(0.0, depth_z)
    (x_min, x_max), (z_min, z_max) = x_range_total, z_range_total
###################################################
# Create the sizing function 
ef_segy2, f_min, f_max = create_sizing_function(
    fname=fname,
    hmin=hmin_segy,
    bbox=segy_bbox,
    wl=wl,
    freq=freq,
    pad_type=padding_type, 
    pad_size_x=padding_x,
    pad_size_z=padding_z,
    grade=grade
)

###################################################
# Generate mesh
print("Generating initial mesh")
sampled_points, tri, boundary_points = generate_mesh(
    x_range=(0.0, length_x),
    z_range=(0.0, depth_z), 
    npoints=npoints,
    density_function=ef_segy2, 
    N=N, 
    pad_type=padding_type,
    ellipse_n=ellipse_n,
    padding_x= padding_x,
    padding_z= padding_z,
    subdomain=subdomain,
    overlap=overlap,
    fineness=fineness,
    f_min=f_min,
    f_max=f_max,
)

end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Initial mesh execution time: {execution_time:.4f} seconds")
sampled_pointsVoronoiCPU = sampled_points
sampled_pointsVoronoiGPU = sampled_points
sampled_pointsPhys = sampled_points
boundary_pointsPhys = boundary_points
boundary_pointsCPU = boundary_points
##############################################################
# Mesh Smoothing
##############################################################
# Physical Smooth
print("Smoothing mesh with physical smooth...")
start_time88=time.perf_counter()
smoothedPhys,triangulationPhys = physical_smooth(
   sampled_pointsPhys.copy(),
   ef_segy2,
   h0=100.0, 
   iterations=physical_iterations, 
   dt=0.05, 
   subdomain=None, 
   ellipse_points=None, 
   boundary_points=boundary_pointsPhys
)
end_time88 = time.perf_counter()
execution_timePhysMesh = end_time88 - start_time88
print(f"Physical smooth execution time: {execution_timePhysMesh:.4f} seconds")

# Export Physical Smooth generated mesh
mesh = meshio.Mesh(
    points=np.column_stack((
    smoothedPhys[:, 0],             
    smoothedPhys[:, 1],             
    np.zeros(len(smoothedPhys)),    
    )),
    cells=[
        ("triangle", np.asarray(triangulationPhys, dtype=np.int32,)),
    ],
)

meshio.write(
    "marmousiPhys.msh",
    mesh,
    file_format="gmsh22",
    binary=False,
)
##############################################################
# CVT CPU Smooth
print("Smoothing mesh with Centroidal Voronoi Tesselation CPU...")
start_time4=time.perf_counter()
smoothedCPU, triangulationCPU = cvt_smooth_cpu(
    sampled_pointsVoronoiCPU.copy(),
    ef_segy2,
    x_min,  # x1
     x_max,      # x2
    z_max, # z1
     z_min,      # z2
    iterations=cvtcpu_iterations,
    influence=1.0,
    hold_boundary=True,
    boundary_points=boundary_pointsCPU
)

end_time4 = time.perf_counter()
execution_timeVOROCPU = end_time4 - start_time4
print(f"Voronoi Tesselation execution time: {execution_timeVOROCPU:.4f} seconds")

# Export CVT CPU generated mesh
mesh = meshio.Mesh(
    points=np.column_stack((
    smoothedCPU[:, 0],             
    smoothedCPU[:, 1],             
    np.zeros(len(smoothedCPU)),    
    )),
    cells=[
        ("triangle", np.asarray(triangulationCPU, dtype=np.int32,)),
    ],
)

meshio.write(
    "marmousiCVTCPU.msh",
    mesh,
    file_format="gmsh22",
    binary=False,
)
##############################################################
# CVT GPU Smooth
start_time44=time.perf_counter()
if CUPY_AVAILABLE:
    print("Smoothing mesh with Centroidal Voronoi Tesselation GPU...")
    smoothedGPU, triangulationGPU = cvt_smooth_gpu(
        sampled_pointsVoronoiGPU,
        ef_segy2,
        x_min,  # x1
         x_max,      # x2
        z_max, # z1
         z_min,      # z2
        cvtgpu_resolution,
        iterations=cvtgpu_iterations,
        influence=1.0,
        hold_boundary=True,
        boundary_points=boundary_points
    )

    # Export CVT GPU generated mesh
    mesh = meshio.Mesh(
        points=np.column_stack((
        smoothedGPU[:, 0],             
        smoothedGPU[:, 1],             
        np.zeros(len(smoothedGPU)),    
        )),
        cells=[
            ("triangle", np.asarray(triangulationGPU, dtype=np.int32,)),
        ],
    )
    
    meshio.write(
        "marmousiCVTGPU.msh",
        mesh,
        file_format="gmsh22",
        binary=False,
    )
else:
    smoothedGPU, triangulationGPU = sampled_points, tri
    print(f"Cupy not available, assigned sampled values.")
    
end_time44 = time.perf_counter()
execution_timeVOROGPU = end_time44 - start_time44
print(f"Voronoi Tesselation execution time: {execution_timeVOROGPU:.4f} seconds")
    
##############################################################
# Laplacian Smooth
print("Smoothing mesh with Laplacian...")
start_time55=time.perf_counter()
smoothedPhyslap,tri = smart_laplacian_smooth_numba(
    smoothedPhys,
    iterations=10, 
    alpha=1.0, 
    boundary_points=boundary_points
)

end_time55 = time.perf_counter()
execution_timeLapPhys = end_time55 - start_time55
print(f"Laplacian for Physical smooth execution time: {execution_timeLapPhys:.4f} seconds")

print("Smoothing mesh with Laplacian...")
start_time56=time.perf_counter()
smoothedCPUlap,tri = smart_laplacian_smooth_numba(
    smoothedCPU,
    iterations=10, 
    alpha=1.0, 
    boundary_points=boundary_points
)

end_time56 = time.perf_counter()
execution_timeLapCPU = end_time56 - start_time56
print(f"Laplacian for CVT CPU execution time: {execution_timeLapCPU:.4f} seconds")

print("Smoothing mesh with Laplacian...")
start_time57=time.perf_counter()
smoothedGPUlap,smoothedGPUlapTri = smart_laplacian_smooth_numba(
    smoothedGPU,
    iterations=10, 
    alpha=1.0, 
    boundary_points=boundary_points
)

end_time57 = time.perf_counter()
execution_timeLapGPU = end_time57 - start_time57
print(f"Laplacian for CVT GPU execution time: {execution_timeLapGPU:.4f} seconds")
##############################################################
# Export CVT GPU + Laplacian generated mesh
mesh = meshio.Mesh(
    points=np.column_stack((
    smoothedGPUlap[:, 0],             
    smoothedGPUlap[:, 1],             
    np.zeros(len(smoothedGPUlap)),    
    )),
    cells=[
        ("triangle", np.asarray(smoothedGPUlapTri, dtype=np.int32,)),
    ],
)

meshio.write(
    "marmousiCVTCGPULAP.msh",
    mesh,
    file_format="gmsh22",
    binary=False,
)

####################################################
# Mesh quality metrics
ntriangles, qmean_sampling, qmin_sampling = get_mesh_stats(sampled_points) 
ntriangles, qmean_samplingPhys, qmin_samplingPhys     = get_mesh_stats(smoothedPhys) 
ntriangles, qmean_samplingCVTCPU, qmin_samplingCVTCPU = get_mesh_stats(smoothedCPU) 
ntriangles, qmean_samplingCVTGPU, qmin_samplingCVTGPU = get_mesh_stats(smoothedGPU) 
ntrianglesPhys, qmean_samplingPhyslap, qmin_samplingPhyslap     = get_mesh_stats(smoothedPhyslap) 
ntrianglesCVTCPU, qmean_samplingCVTCPUlap, qmin_samplingCVTCPUlap = get_mesh_stats(smoothedCPUlap)
ntrianglesCVTGPU, qmean_samplingCVTGPUlap, qmin_samplingCVTGPUlap = get_mesh_stats(smoothedGPUlap)

# Element size deviation related to the velocity model
s0,_ = mesh_sizing_check(sampled_points, ef_segy2)
s1,_ = mesh_sizing_check(smoothedPhys, ef_segy2)
s2,_ = mesh_sizing_check(smoothedCPU, ef_segy2)
s3,_ = mesh_sizing_check(smoothedGPU, ef_segy2)
s4,_ = mesh_sizing_check(smoothedPhyslap, ef_segy2)
s5,_ = mesh_sizing_check(smoothedCPUlap, ef_segy2)
s6,_ = mesh_sizing_check(smoothedGPUlap, ef_segy2)

###############################################################
# Results
print("\nMESH DEVIATION")
print(f"{'Sampling':>15} {'Physical':>15} {'CVT CPU':>15} {'CVT GPU':>15} "
      f"{'Phys+Lap':>15} {'CPU+Lap':>15} {'GPU+Lap':>15}")

print(f"{s0:15.6f} {s1:15.6f} {s2:15.6f} {s3:15.6f} "
      f"{s4:15.6f} {s5:15.6f} {s6:15.6f}")


print("\nEXECUTION TIMES")
print(f"{'Sampling':>15} {'Physical':>15} {'CVT CPU':>15} {'CVT GPU':>15} "
      f"{'Phys+Lap':>15} {'CPU+Lap':>15} {'GPU+Lap':>15}")

print(f"{execution_time:15.6f} "
      f"{execution_timePhysMesh:15.6f} "
      f"{execution_timeVOROCPU:15.6f} "
      f"{execution_timeVOROGPU:15.6f} "
      f"{execution_time + execution_timePhysMesh + execution_timeLapPhys:15.6f} "
      f"{execution_time + execution_timeVOROCPU + execution_timeLapCPU:15.6f} "
      f"{execution_time + execution_timeVOROGPU + execution_timeLapGPU:15.6f}")


print("\nMESH QUALITY")
print(f"{'Method':>15} {'Triangles':>12} {'Initial mean':>15} {'Initial min':>15} "
      f"{'Smooth mean':>15} {'Smooth min':>15} {'Lap mean':>15} {'Lap min':>15}")

print(f"{'Physical':>15} {ntrianglesPhys:12d} "
      f"{qmean_sampling:15.6f} {qmin_sampling:15.6f} "
      f"{qmean_samplingPhys:15.6f} {qmin_samplingPhys:15.6f} "
      f"{qmean_samplingPhyslap:15.6f} {qmin_samplingPhyslap:15.6f}")

print(f"{'CVT CPU':>15} {ntrianglesCVTCPU:12d} "
      f"{qmean_sampling:15.6f} {qmin_sampling:15.6f} "
      f"{qmean_samplingCVTCPU:15.6f} {qmin_samplingCVTCPU:15.6f} "
      f"{qmean_samplingCVTCPUlap:15.6f} {qmin_samplingCVTCPUlap:15.6f}")

print(f"{'CVT GPU':>15} {ntrianglesCVTGPU:12d} "
      f"{qmean_sampling:15.6f} {qmin_sampling:15.6f} "
      f"{qmean_samplingCVTGPU:15.6f} {qmin_samplingCVTGPU:15.6f} "
      f"{qmean_samplingCVTGPUlap:15.6f} {qmin_samplingCVTGPUlap:15.6f}")
######################################################



######################################################
# Plotting 

print("Bubble Sampling Plot")
plot_mesh_circles(points=sampled_points,f_centers_fn=ef_segy2,x_min=x_min, x_max=x_max, z_min=z_max, z_max=z_min)

print("Sizing Function Plot")
plot_mesh(smoothedGPU, x_range_total, z_range_total, 
                  ef_segy2, show_points=False, show_density=True,show_element_size=False, filename=f"sizing_{freq}_")

print("CVT CPU Plot")
plot_mesh(smoothedCPU, x_range_total, z_range_total, 
                  ef_segy2, show_points=False, show_density=False,show_element_size=False, filename=f"voronoiGPU_{freq}_")

print("CVT CPU - Voronoi Diagram Plot")
plot_fast_voronoi(smoothedCPU)

if CUPY_AVAILABLE:
    print("CVT GPU Plot")
    plot_mesh(smoothedGPU, x_range_total, z_range_total, 
                      ef_segy2, show_points=False, show_density=False,show_element_size=False, filename=f"voronoiGPU_{freq}_")
    
    print("CVT GPU + Laplacian Plot")
    plot_mesh(smoothedGPUlap, x_range_total, z_range_total, 
                      ef_segy2, show_points=False, show_density=False,show_element_size=False, filename=f"voronoiCPULAPLACIAN_{freq}_")
#######################################################




## 